In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
from scipy.io import loadmat
import os
import gc

# Set reshape = 1 if you are reshaping and saving the arrays for the first time. Set reshape = 0 if you have already done so and would just like to load them in.
reshape = 1

Load in EEG Rest Data

In [ ]:
if reshape:

    trim_size = 7

    # Subjects that exceed translation/rotation parameters
    eeg_subjects_toremove = [1,7]

    # --- Load arousal (usually smaller) ---
    spectral_slope_array = np.load("../4_results/1_arousal/1_EEG_Rest/Convolved_EEG_Rest_Arousal.npy")
    print(f"Arousal Data Size: {spectral_slope_array.shape}")

    # --- Load FC with memmap (avoid pulling full file into RAM immediately) ---
    eeg_fc = np.load(
        "../4_results/1_arousal/1_EEG_Rest/EEG_FC_array.npy",
        mmap_mode="r",
    )  # (n_subj, n_runs, n_TRs_original, n_features)

    n_subj, n_runs, n_TRs_original, n_features = eeg_fc.shape
    n_TRs_trimmed = n_TRs_original - 2 * trim_size
    total_TRs_trimmed = n_runs * n_TRs_trimmed

    # Trim via slicing (view over memmap)
    eeg_fc_trim_view = eeg_fc[:, :, trim_size:-trim_size, :]

    # Remove subjects BEFORE making a contiguous reshaped array (saves RAM)
    keep_mask = np.ones(n_subj, dtype=bool)
    keep_mask[eeg_subjects_toremove] = False

    eeg_array_cut = spectral_slope_array[keep_mask].copy()
    del spectral_slope_array

    # Materialize only kept subjects once, then reshape
    eeg_fc_kept = np.array(eeg_fc_trim_view[keep_mask], copy=True)
    eeg_fc_array_cut = eeg_fc_kept.reshape(eeg_fc_kept.shape[0], total_TRs_trimmed, n_features)

    # Free intermediates aggressively
    del eeg_fc, eeg_fc_trim_view, eeg_fc_kept, keep_mask

    print(f"EEG arousal cut shape: {eeg_array_cut.shape}")
    print(f"EEG FC cut shape: {eeg_fc_array_cut.shape}")

    np.save(
        "../4_results/1_arousal/1_EEG_Rest/EEG_Rest_FC_Cut.npy", eeg_fc_array_cut)

    np.save(
        "../4_results/1_arousal/1_EEG_Rest/EEG_Rest_Arousal_Cut.npy", eeg_array_cut)

Load in Pupil Rest Data

In [ ]:
if reshape:

    trim_size = 15
    # Subjects that exceed translation/rotation parameters
    pupil_subjects_toremove = [2, 18]

    # --- Load pupil (usually smaller) ---
    pupil_array = np.load("../4_results/1_arousal/2_Pupil_Rest/Convolved_Pupil_Rest_Arousal.npy")
    print(f"Pupil arousal data shape: {pupil_array.shape}")

    # --- Load FC with memmap (doesn't fully load into RAM) ---
    pupil_fc = np.load(
        "../4_results/1_arousal/2_Pupil_Rest/Pupil_FC_array.npy",
        mmap_mode="r",
    )  # shape: (n_subj, n_runs, n_TRs_original, n_features)

    n_subj, n_runs, n_TRs_original, n_features = pupil_fc.shape
    n_TRs_trimmed = n_TRs_original - 2 * trim_size
    total_TRs_trimmed = n_runs * n_TRs_trimmed

    # Trim using slicing (still a view over the memmap)
    pupil_fc_trim_view = pupil_fc[:, :, trim_size:-trim_size, :]

    # Remove subjects BEFORE making a contiguous reshaped array (saves RAM)
    keep_mask = np.ones(n_subj, dtype=bool)
    keep_mask[pupil_subjects_toremove] = False

    pupil_array_cut = pupil_array[keep_mask].copy()  # small-ish; make it standalone
    del pupil_array
    gc.collect()

    # Materialize only the kept subjects as a contiguous array, then reshape
    # (reshape after copy ensures correct memory layout)
    pupil_fc_kept = np.array(pupil_fc_trim_view[keep_mask], copy=True)  # big allocation happens once
    pupil_fc_array_cut = pupil_fc_kept.reshape(pupil_fc_kept.shape[0], total_TRs_trimmed, n_features)

    # Free intermediates aggressively
    del pupil_fc, pupil_fc_trim_view, pupil_fc_kept, keep_mask
    gc.collect()

    print(f"Pupil arousal cut shape: {pupil_array_cut.shape}")
    print(f"Pupil FC cut shape: {pupil_fc_array_cut.shape}")

    np.save(
        "../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_FC_Cut.npy", pupil_fc_array_cut)
    np.save(
        "../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_Arousal_Cut.npy", pupil_array_cut)

Load in Pupil Story

In [ ]:
if reshape:

    # Subjects that exceed translation/rotation parameters
    paranoia_subjects_toremove = [0, 20]

    # --- Load paranoia arousal ---
    paranoia_base = np.load("../4_results/1_arousal/3_Pupil_Story/Convolved_Pupil_Story_Arousal.npy")

    # Tile once, then immediately apply subject mask to avoid extra copies later
    n_subj = 22
    paranoia_array = np.tile(paranoia_base, (n_subj, 1))

    print(f"Paranoia arousal data shape (pre-cut): {paranoia_array.shape}")

    # --- Load FC with memmap (already trimmed, so no slicing needed) ---
    paranoia_fc = np.load(
        "../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_array.npy",
        mmap_mode="r",
    )

    print(f"Paranoia FC data shape (pre-cut): {paranoia_fc.shape}")

    # Build keep mask
    keep_mask = np.ones(n_subj, dtype=bool)
    keep_mask[paranoia_subjects_toremove] = False

    # Cut arousal (small → safe to copy)
    paranoia_array_cut = paranoia_array[keep_mask].copy()
    del paranoia_array, paranoia_base

    # Cut FC (materialize only kept subjects once)
    paranoia_fc_array_cut = np.array(paranoia_fc[keep_mask], copy=True)

    # Free intermediates
    del paranoia_fc, keep_mask

    print(f"Paranoia arousal cut shape: {paranoia_array_cut.shape}")
    print(f"Paranoia FC cut shape: {paranoia_fc_array_cut.shape}")

    # Final outputs:
    # paranoia_array_cut, paranoia_fc_array_cut

    np.save(
        "../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_Cut.npy", paranoia_fc_array_cut)
    np.save(
        "../4_results/1_arousal/3_Pupil_Story/Pupil_Story_Arousal_Cut.npy", paranoia_array_cut)

Load in FilmFestival

In [ ]:
if reshape:

    # Subjects that exceed translation/rotation parameters
    FF_subjects_toremove = [2,3,4,5,14]

    # --- Load FilmFest event arousal ---
    FF_event_base = np.load("../4_results/1_arousal/4_FilmFestival/Filmfest_Event_Arousal.npy")

    n_subj = 20
    FF_event_array = np.tile(FF_event_base, (n_subj, 1))

    print(f"FilmFest event arousal data shape (pre-cut): {FF_event_array.shape}")

    # --- Load FC with memmap to limit RAM usage ---
    FF_event_fc = np.load(
        "../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_array.npy",
        mmap_mode="r",
    )

    print(f"FilmFest event FC data shape (pre-cut): {FF_event_fc.shape}")

    # Build keep mask
    keep_mask = np.ones(n_subj, dtype=bool)
    keep_mask[FF_subjects_toremove] = False

    # Cut arousal (small → safe to copy)
    FF_event_array_cut = FF_event_array[keep_mask].copy()
    del FF_event_array, FF_event_base

    # Cut FC (materialize only kept subjects once)
    FF_event_fc_array_cut = np.array(FF_event_fc[keep_mask], copy=True)

    # Free intermediates
    del FF_event_fc, keep_mask

    print(f"FilmFest event arousal cut shape: {FF_event_array_cut.shape}")
    print(f"FilmFest event FC cut shape: {FF_event_fc_array_cut.shape}")

    # Final outputs:
    # FF_event_array_cut, FF_event_fc_array_cut

    np.save(
        "../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_Cut.npy", FF_event_fc_array_cut)
    np.save(
        "../4_results/1_arousal/4_FilmFestival/FilmFestival_Arousal_Cut.npy", FF_event_array_cut)

Load in Sherlock

In [ ]:
if reshape:

    # Subjects that exceed translation/rotation parameters
    Sherlock_subjects_toremove = [12]

    # --- Load Sherlock event arousal ---
    Sherlock_event_base = np.load("../4_results/1_arousal/5_Sherlock/Sherlock_Event_Arousal.npy")

    n_subj = 15
    Sherlock_event_array = np.tile(Sherlock_event_base, (n_subj, 1))

    print(f"Sherlock event arousal data shape (pre-cut): {Sherlock_event_array.shape}")

    # --- Load FC with memmap to reduce RAM pressure ---
    Sherlock_event_fc = np.load(
        "../4_results/1_arousal/5_Sherlock/Sherlock_FC_array.npy",
        mmap_mode="r",
    )

    print(f"Sherlock event FC data shape (pre-cut): {Sherlock_event_fc.shape}")

    # Build keep mask
    keep_mask = np.ones(n_subj, dtype=bool)
    keep_mask[Sherlock_subjects_toremove] = False

    # Cut arousal (small → safe to copy)
    Sherlock_event_array_cut = Sherlock_event_array[keep_mask].copy()
    del Sherlock_event_array, Sherlock_event_base

    # Cut FC (materialize only kept subjects once)
    Sherlock_event_fc_array_cut = np.array(Sherlock_event_fc[keep_mask], copy=True)

    # Free intermediates
    del Sherlock_event_fc, keep_mask

    print(f"Sherlock event arousal cut shape: {Sherlock_event_array_cut.shape}")
    print(f"Sherlock event FC cut shape: {Sherlock_event_fc_array_cut.shape}")

    # Final outputs:
    # Sherlock_event_array_cut, Sherlock_event_fc_array_cut

    np.save(
        "../4_results/1_arousal/5_Sherlock/Sherlock_FC_Cut.npy", Sherlock_event_fc_array_cut)
    np.save(
        "../4_results/1_arousal/5_Sherlock/Sherlock_Arousal_Cut.npy", Sherlock_event_array_cut)

In [ ]:
if ~reshape:
    eeg_fc_array_cut = np.load("../4_results/1_arousal/1_EEG_Rest/EEG_Rest_FC_Cut.npy")
    eeg_array_cut = np.load("../4_results/1_arousal/1_EEG_Rest/EEG_Rest_Arousal_Cut.npy")
    pupil_fc_array_cut = np.load("../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_FC_Cut.npy")
    pupil_array_cut = np.load("../4_results/1_arousal/2_Pupil_Rest/Pupil_Rest_Arousal_Cut.npy")
    paranoia_fc_array_cut = np.load("../4_results/1_arousal/3_Pupil_Story/Pupil_Story_FC_Cut.npy")
    paranoia_array_cut = np.load("../4_results/1_arousal/3_Pupil_Story/Pupil_Story_Arousal_Cut.npy")
    FF_event_array_cut = np.load("../4_results/1_arousal/4_FilmFestival/FilmFestival_Arousal_Cut.npy")
    FF_event_fc_array_cut = np.load("../4_results/1_arousal/4_FilmFestival/FilmFestival_FC_Cut.npy")
    Sherlock_event_array_cut = np.load("../4_results/1_arousal/5_Sherlock/Sherlock_Arousal_Cut.npy")
    Sherlock_event_fc_array_cut = np.load("../4_results/1_arousal/5_Sherlock/Sherlock_FC_Cut.npy")

Function to perform CPM prediction across datasets

In [ ]:
def cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path,
                           alpha_threshold=0.05, use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False):
    """
    Perform cross-modal prediction of arousal data using functional connectivity features.

    Parameters:
    - arousal1_array: np.array, Target arousal data (n_subj1, n_TRs1)
    - fc1_array: np.array, Target FC data (n_subj1, n_TRs1, n_features)
    - arousal2_array: np.array, Source arousal data (n_subj2, n_TRs2)
    - fc2_array: np.array, Source FC data (n_subj2, n_TRs2, n_features)
    - predicted_output_path: str, File path to save predicted arousal1 array
    - observed_output_path: str, File path to save observed arousal1 array
    - features_output_path: str, File path to save significant features CSV
    - alpha_threshold: float, Significance threshold for feature selection (default=0.05)
    - use_mean: bool, If True, use mean; if False, use median for feature weights.
    - use_binary_weights: bool, If True, use binary weights (1, -1); if False, use actual weights.
    - normalize_by_total: bool, If True, normalize by total significant features; if False, normalize by separate counts.
    - use_absolute: bool, If True, use absolute value sum for positive and negative features.
    """


    print(f"Shape of Arousal 1 data: {arousal1_array.shape}")
    print(f"Shape of Arousal 1 FC data: {fc1_array.shape}")
    print(f"Shape of Arousal 2 data: {arousal2_array.shape}")
    print(f"Shape of Arousal 2 FC data: {fc2_array.shape}")

    # ---- Step 2: Initialize Storage ---- #
    n_subj1, n_TRs1, n_features = fc1_array.shape
    n_subj2, n_TRs2 = arousal2_array.shape

    predicted_arousal1 = np.full((n_subj1, n_TRs1), np.nan)
    observed_arousal1 = arousal1_array

    # ---- Step 3: Feature Extraction from Arousal 2 ---- #
    print("Extracting features from Arousal 2 data...")
    r_matrix = np.zeros((n_subj2, n_features))

    for subj_idx in range(n_subj2):
        fc2 = fc2_array[subj_idx]
        arousal2 = arousal2_array[subj_idx]

        valid_mask = (~np.isnan(arousal2) & ~np.isnan(fc2).any(axis=1))
        fc2_clean = fc2[valid_mask]
        arousal2_clean = arousal2[valid_mask]

        for feature_idx in range(n_features):
            r, _ = stats.pearsonr(fc2_clean[:, feature_idx], arousal2_clean)
            r_matrix[subj_idx, feature_idx] = r

    # ---- Step 4: One-Sample t-Test Across Arousal 2 Subjects ---- #
    print("Running one-sample t-test on Arousal 2 features...")
    t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')

    # Use mean or median as feature weights
    if use_mean:
        feature_weights = np.mean(r_matrix, axis=0)
    else:
        feature_weights = np.median(r_matrix, axis=0)

    significant_features = np.where(p_values < alpha_threshold)[0]

    # Create binary mask or keep actual weights
    if use_binary_weights:
        binary_mask = np.sign(feature_weights[significant_features])
        weight_mask = binary_mask  # Use only -1, 0, 1
        print(weight_mask)
    else:
        weight_mask = feature_weights[significant_features]  # Keep original values
        print(weight_mask)

    num_positive = np.sum(weight_mask > 0)
    num_negative = np.sum(weight_mask < 0)
    total_features = len(significant_features)

    print(f"{len(significant_features)} significant features selected (p < {alpha_threshold})")
    print(f"Number of positive features: {num_positive}")
    print(f"Number of negative features: {num_negative}")
    print(f"Total number of features: {total_features}")

    # ---- Step 5: Map Features to ROI Pairs ---- #
    print("Mapping significant features to ROI pairs...")
    n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
    sig_roi_pairs = [roi_pairs[idx] for idx in significant_features]

    significant_df = pd.DataFrame({
        'ROI1': [pair[0] for pair in sig_roi_pairs],
        'ROI2': [pair[1] for pair in sig_roi_pairs],
        'p_value': p_values[significant_features],
        'feature_weight': feature_weights[significant_features]
    })

    # ---- Step 6: Apply Extracted Features to Arousal 1 Data ---- #
    print("Applying Arousal 2-derived features to Arousal 1 data...")

    for subj_idx in range(n_subj1):
        fc1 = fc1_array[subj_idx][:, significant_features]
        masked_fc1 = fc1 * weight_mask  # Apply weights (binary or real)

        arousal_per_TR = np.full(n_TRs1, np.nan)
        for i in range(n_TRs1):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weight_mask > 0]))
                sum_negative = np.sum(np.abs(fc_values[weight_mask < 0]))
            else:
                sum_positive = np.sum(fc_values[weight_mask > 0])
                sum_negative = np.sum(fc_values[weight_mask < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        predicted_arousal1[subj_idx, :] = arousal_per_TR

    # ---- Step 7: Save Results ---- #
    print("Saving cross-modal prediction results...")

    # Ensure output directories exist
    os.makedirs(os.path.dirname(predicted_output_path), exist_ok=True)
    os.makedirs(os.path.dirname(observed_output_path), exist_ok=True)
    os.makedirs(os.path.dirname(features_output_path), exist_ok=True)

    np.save(predicted_output_path, predicted_arousal1)
    np.save(observed_output_path, observed_arousal1)
    significant_df.to_csv(features_output_path, index=False)

    print("✅ Cross-modal prediction completed! 🚀")


Function to perform CPM within dataset; used for EEG Rest and Pupil Rest

In [ ]:
def cross_modal_prediction_loso(arousal_array, fc_array,
                                predicted_output_path, observed_output_path, features_output_path,
                                alpha_threshold=0.05, use_mean=True, use_binary_weights=True,
                                normalize_by_total=True, use_absolute=False):
    """
    Leave-One-Subject-Out cross-modal prediction of arousal using FC features.
    Each subject is predicted using models trained on all other subjects.

    Parameters:
    - arousal_array: np.array, shape (n_subj, n_TRs)
    - fc_array: np.array, shape (n_subj, n_TRs, n_features)
    - predicted_output_path: str
    - observed_output_path: str
    - features_output_path: str
    - alpha_threshold: float
    - use_mean: bool
    - use_binary_weights: bool
    - normalize_by_total: bool
    - use_absolute: bool
    """

    n_subj, n_TRs, n_features = fc_array.shape
    predicted_arousal = np.full((n_subj, n_TRs), np.nan)
    observed_arousal = arousal_array.copy()
    all_feature_logs = []

    for test_subj in range(n_subj):
        print(f"\n🔁 Leaving out subject {test_subj + 1}/{n_subj}...")

        # Split into training subjects
        train_idx = [i for i in range(n_subj) if i != test_subj]
        n_train_subj = len(train_idx)

        # Step 1: Per-subject correlations → r_matrix shape: (n_train_subj, n_features)
        r_matrix = np.full((n_train_subj, n_features), np.nan)

        for i, subj in enumerate(train_idx):
            fc_subj = fc_array[subj]
            arousal_subj = arousal_array[subj]

            valid_mask = (~np.isnan(arousal_subj)) & (~np.isnan(fc_subj).any(axis=1))
            fc_clean = fc_subj[valid_mask]
            arousal_clean = arousal_subj[valid_mask]

            if len(arousal_clean) < 3:
                continue  # skip if too short

            for feat_idx in range(n_features):
                r, _ = stats.pearsonr(fc_clean[:, feat_idx], arousal_clean)
                r_matrix[i, feat_idx] = r

        # Step 2: One-sample t-test across subjects
        t_vals, p_vals = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')
        feature_weights = np.mean(r_matrix, axis=0) if use_mean else np.median(r_matrix, axis=0)
        significant_features = np.where(p_vals < alpha_threshold)[0]

        if len(significant_features) == 0:
            print(f"⚠️ No significant features for subject {test_subj + 1}. Skipping prediction.")
            continue

        # Step 3: Create weights
        weights = np.sign(feature_weights[significant_features]) if use_binary_weights else feature_weights[significant_features]
        num_positive = np.sum(weights > 0)
        num_negative = np.sum(weights < 0)
        total_features = len(significant_features)

        print(f"✅ Selected {total_features} significant features (p < {alpha_threshold})")

        # Step 4: Map features to ROI pairs
        n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
        roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
        sig_roi_pairs = [roi_pairs[idx] for idx in significant_features]

        feature_df = pd.DataFrame({
            'Subject': test_subj + 1,
            'ROI1': [pair[0] for pair in sig_roi_pairs],
            'ROI2': [pair[1] for pair in sig_roi_pairs],
            'p_value': p_vals[significant_features],
            'feature_weight': feature_weights[significant_features]
        })
        all_feature_logs.append(feature_df)

        # Step 5: Predict arousal for test subject
        fc_test = fc_array[test_subj][:, significant_features]
        arousal_pred = np.full(n_TRs, np.nan)

        for t in range(n_TRs):
            if np.isnan(fc_test[t]).any():
                continue

            if use_absolute:
                sum_pos = np.sum(np.abs(fc_test[t][weights > 0]))
                sum_neg = np.sum(np.abs(fc_test[t][weights < 0]))
            else:
                sum_pos = np.sum(fc_test[t][weights > 0])
                sum_neg = np.sum(fc_test[t][weights < 0])

            if normalize_by_total:
                divisor = total_features
            else:
                divisor_pos = num_positive if num_positive > 0 else 1
                divisor_neg = num_negative if num_negative > 0 else 1

            avg_pos = sum_pos / divisor if normalize_by_total else sum_pos / divisor_pos
            avg_neg = sum_neg / divisor if normalize_by_total else sum_neg / divisor_neg

            arousal_pred[t] = avg_pos - avg_neg

        predicted_arousal[test_subj] = arousal_pred

    os.makedirs(os.path.dirname(predicted_output_path), exist_ok=True)


    # Step 6: Save results
    np.save(predicted_output_path, predicted_arousal)
    np.save(observed_output_path, observed_arousal)

    all_features_df = pd.concat(all_feature_logs, ignore_index=True)
    all_features_df.to_csv(features_output_path, index=False)

    print("\n✅ LOSO cross-modal prediction completed.")



Movies Features and Prediction

In [ ]:
def compute_subject_feature_correlations(arousal_array, fc_array):
    """
    Compute per-subject correlations between FC features and arousal.

    Parameters:
    - arousal_array: shape (n_subj, n_TRs)
    - fc_array: shape (n_subj, n_TRs, n_features)

    Returns:
    - r_matrix: shape (n_subj, n_features)
    """
    n_subj, _, n_features = fc_array.shape
    r_matrix = np.full((n_subj, n_features), np.nan)

    for subj_idx in range(n_subj):
        arousal = arousal_array[subj_idx]
        fc = fc_array[subj_idx]

        valid_mask = (~np.isnan(arousal) & ~np.isnan(fc).any(axis=1))
        arousal_clean = arousal[valid_mask]
        fc_clean = fc[valid_mask]

        for feat_idx in range(n_features):
            if len(arousal_clean) > 1:
                r, _ = stats.pearsonr(fc_clean[:, feat_idx], arousal_clean)
                r_matrix[subj_idx, feat_idx] = r

    return r_matrix


def select_and_save_significant_features_v2(
    arousal_sherlock, fc_sherlock,
    arousal_filmfest, fc_filmfest,
    output_csv_path,
    alpha_threshold=0.05,
    use_mean=True
):
    """
    Computes subject-wise correlations within each dataset, combines them,
    and saves significant features (based on t-test) to CSV.

    Parameters:
    - arousal_sherlock: np.array (n_subj_sher, n_TR_sher)
    - fc_sherlock: np.array (n_subj_sher, n_TR_sher, n_features)
    - arousal_filmfest: np.array (n_subj_film, n_TR_film)
    - fc_filmfest: np.array (n_subj_film, n_TR_film, n_features)
    - output_csv_path: str, path to save CSV
    - alpha_threshold: float, p-value cutoff for significance
    - use_mean: bool, use mean (True) or median (False) for feature weights
    """

    print("Computing per-subject correlations...")
    r_sher = compute_subject_feature_correlations(arousal_sherlock, fc_sherlock)
    r_film = compute_subject_feature_correlations(arousal_filmfest, fc_filmfest)

    print("Combining datasets...")
    r_matrix = np.vstack([r_sher, r_film])  # shape (n_total_subj, n_features)
    n_features = r_matrix.shape[1]

    print("Running one-sample t-test across all subjects...")
    t_values, p_values = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')
    feature_weights = np.mean(r_matrix, axis=0) if use_mean else np.median(r_matrix, axis=0)

    significant_features = np.where(p_values < alpha_threshold)[0]
    print(f"{len(significant_features)} features passed p < {alpha_threshold}")

    print("Mapping significant features to ROI pairs...")
    n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
    selected_pairs = [roi_pairs[i] for i in significant_features]

    df = pd.DataFrame({
        'ROI1': [pair[0] for pair in selected_pairs],
        'ROI2': [pair[1] for pair in selected_pairs],
        'p_value': p_values[significant_features],
        'feature_weight': feature_weights[significant_features]
    })

    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

    df.to_csv(output_csv_path, index=False)
    print(f"✅ Saved {len(df)} significant features to {output_csv_path}")

    return df, significant_features, feature_weights

df_sig, SR_features, SR_weights = select_and_save_significant_features_v2(
    Sherlock_event_array_cut, Sherlock_event_fc_array_cut,
    FF_event_array_cut, FF_event_fc_array_cut,
    output_csv_path="../4_results/2_classification/1_Movies_within/Movies_features.csv",
    alpha_threshold=0.05,
    use_mean=True
)

def SR_to_other(test_arousal, test_fc_array, pop, oop, use_absolute=False, normalize_by_total=True):

    n_subj = test_fc_array.shape[0]
    n_TRs = test_fc_array.shape[1]
    n_features = test_fc_array.shape[2]
    total_features = len(SR_features)
    num_positive = np.sum(SR_binarized > 0)
    num_negative = np.sum(SR_binarized < 0)
    predicted_arousal = np.full((n_subj, n_TRs), np.nan)

    for subj_idx in range(n_subj):
        fc = test_fc_array[subj_idx][:, SR_features]
        masked_fc = fc * SR_binarized  # Apply weights (binary or real)

        arousal_per_TR = np.full(n_TRs, np.nan)
        for i in range(n_TRs):
            fc_values = masked_fc[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[SR_binarized > 0]))
                sum_negative = np.sum(np.abs(fc_values[SR_binarized < 0]))
            else:
                sum_positive = np.sum(fc_values[SR_binarized > 0])
                sum_negative = np.sum(fc_values[SR_binarized < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        predicted_arousal[subj_idx, :] = arousal_per_TR

    os.makedirs(os.path.dirname(pop), exist_ok=True)
    np.save(pop, predicted_arousal)
    np.save(oop, test_arousal)
    
SR_binarized = np.sign(SR_weights[SR_features])
np.save("../4_results/2_classification/1_Movies_within/Movies_Features_Binarized.npy", SR_binarized)

def other_to_SR(sherlock_fc, sherlock_beh, filmfest_fc, filmfest_beh, other_fc, other_beh, sherpop, sheroop, ffpop, ffoop, features_output_path, alpha_threshold=0.05, use_mean=True, use_binary_weights=True, use_absolute = False, normalize_by_total = True):

    all_feature_logs = []
    other_nsubj = other_fc.shape[0]
    n_features = other_fc.shape[2]
    other_TR = other_fc.shape[1]

    sher_nsubj = sherlock_fc.shape[0]
    ff_nsubj = filmfest_fc.shape[0]

    sher_ntr = sherlock_fc.shape[1]
    ff_ntr = filmfest_fc.shape[1]
    

    r_matrix = np.zeros((other_nsubj, n_features))

    for subj_idx in range(other_nsubj):
        this_other_fc = other_fc[subj_idx]
        this_other_beh = other_beh[subj_idx]

        valid_mask = (~np.isnan(this_other_beh) & ~np.isnan(this_other_fc).any(axis=1))
        fc2_clean = this_other_fc[valid_mask]
        arousal2_clean = this_other_beh[valid_mask]

        for feature_idx in range(n_features):
            r, _ = stats.pearsonr(fc2_clean[:, feature_idx], arousal2_clean)
            r_matrix[subj_idx, feature_idx] = r

    t_vals, p_vals = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy='omit')
    feature_weights = np.mean(r_matrix, axis=0) if use_mean else np.median(r_matrix, axis=0)
    significant_features = np.where(p_vals < alpha_threshold)[0]


    weights = np.sign(feature_weights[significant_features]) if use_binary_weights else feature_weights[significant_features]
    num_positive = np.sum(weights > 0)
    num_negative = np.sum(weights < 0)
    total_features = len(significant_features)


    # ---- Step 5: Map Features to ROI Pairs ---- #
    print("Mapping significant features to ROI pairs...")
    n_ROIs = int((1 + np.sqrt(1 + 8 * n_features)) / 2)
    roi_pairs = list(combinations(range(1, n_ROIs + 1), 2))
    sig_roi_pairs = [roi_pairs[idx] for idx in significant_features]

    significant_df = pd.DataFrame({
        'ROI1': [pair[0] for pair in sig_roi_pairs],
        'ROI2': [pair[1] for pair in sig_roi_pairs],
        'p_value': p_vals[significant_features],
        'feature_weight': feature_weights[significant_features]
    })
    all_feature_logs.append(significant_df)

    print("Applying Arousal 2-derived features to Movie data...")

    ff_predicted = np.full((ff_nsubj, ff_ntr), np.nan)
    sher_predicted = np.full((sher_nsubj, sher_ntr), np.nan)

    for subj_idx in range(ff_nsubj):
        fc1 = filmfest_fc[subj_idx][:, significant_features]
        masked_fc1 = fc1 * weights  

        arousal_per_TR = np.full(ff_ntr, np.nan)
        for i in range(ff_ntr):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weights > 0]))
                sum_negative = np.sum(np.abs(fc_values[weights < 0]))
            else:
                sum_positive = np.sum(fc_values[weights > 0])
                sum_negative = np.sum(fc_values[weights < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR[i] = avg_positive - avg_negative

        ff_predicted[subj_idx, :] = arousal_per_TR

    for subj_idx in range(sher_nsubj):
        fc1 = sherlock_fc[subj_idx][:, significant_features]
        masked_fc1 = fc1 * weights  

        arousal_per_TR_2 = np.full(sher_ntr, np.nan)
        for i in range(sher_ntr):
            fc_values = masked_fc1[i, :]

            if use_absolute:
                sum_positive = np.sum(np.abs(fc_values[weights > 0]))
                sum_negative = np.sum(np.abs(fc_values[weights < 0]))
            else:
                sum_positive = np.sum(fc_values[weights > 0])
                sum_negative = np.sum(fc_values[weights < 0])

            if normalize_by_total:
                divisor = total_features if total_features > 0 else 1
            else:
                divisor_positive = num_positive if num_positive > 0 else 1
                divisor_negative = num_negative if num_negative > 0 else 1

            avg_positive = sum_positive / divisor if normalize_by_total else sum_positive / divisor_positive
            avg_negative = sum_negative / divisor if normalize_by_total else sum_negative / divisor_negative

            arousal_per_TR_2[i] = avg_positive - avg_negative

        sher_predicted[subj_idx, :] = arousal_per_TR_2
    
    os.makedirs(os.path.dirname(ffpop), exist_ok=True)
    np.save(ffpop, ff_predicted)
    np.save(sherpop, sher_predicted)
    np.save(ffoop, filmfest_beh)
    np.save(sheroop, sherlock_beh)

    all_features_df = pd.concat(all_feature_logs, ignore_index=True)
    all_features_df.to_csv(features_output_path, index=False)

    


In [ ]:
arousal1_array = FF_event_array_cut
fc1_array = FF_event_fc_array_cut
arousal2_array = Sherlock_event_array_cut
fc2_array = Sherlock_event_fc_array_cut

predicted_output_path = "../4_results/2_classification/1_Movies_within/FF_from_Sherlock_predicted.npy"
observed_output_path = "../4_results/2_classification/1_Movies_within/FF_from_Sherlock_observed.npy"
features_output_path = "../4_results/2_classification/1_Movies_within/FF_from_Sherlock_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



arousal2_array = FF_event_array_cut
fc2_array = FF_event_fc_array_cut
arousal1_array = Sherlock_event_array_cut
fc1_array = Sherlock_event_fc_array_cut

predicted_output_path = "../4_results/2_classification/1_Movies_within/Sherlock_from_FF_predicted.npy"
observed_output_path = "../4_results/2_classification/1_Movies_within/Sherlock_from_FF_observed.npy"
features_output_path = "../4_results/2_classification/1_Movies_within/Sherlock_from_FF_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



Predicting EEG Rest within

In [ ]:
arousal_data = eeg_array_cut
fc_data = eeg_fc_array_cut
predicted_output_path = "../4_results/2_classification/2_EEG_Rest_within/EEG_Rest_within_predicted.npy"
observed_output_path = "../4_results/2_classification/2_EEG_Rest_within/EEG_Rest_within_observed.npy"
features_output_path = "../4_results/2_classification/2_EEG_Rest_within/EEG_Rest_within_features.csv"

cross_modal_prediction_loso(
    arousal_array=arousal_data,              # shape (n_subjects, n_TRs)
    fc_array=fc_data,                        # shape (n_subjects, n_TRs, n_features)
    predicted_output_path=predicted_output_path,
    observed_output_path=observed_output_path,
    features_output_path=features_output_path,
    alpha_threshold=0.05,
    use_mean=True,
    use_binary_weights=True,
    normalize_by_total=True,
    use_absolute=False
)


Pupil Rest within

In [ ]:
arousal_data = pupil_array_cut
fc_data = pupil_fc_array_cut
predicted_output_path = "../4_results/2_classification/3_Pupil_Rest_within/Pupil_Rest_within_predicted.npy"
observed_output_path = "../4_results/2_classification/3_Pupil_Rest_within/Pupil_Rest_within_observed.npy"
features_output_path = "../4_results/2_classification/3_Pupil_Rest_within/Pupil_Rest_within_features.csv"

cross_modal_prediction_loso(
    arousal_array=arousal_data,              # shape (n_subjects, n_TRs)
    fc_array=fc_data,                        # shape (n_subjects, n_TRs, n_features)
    predicted_output_path=predicted_output_path,
    observed_output_path=observed_output_path,
    features_output_path=features_output_path,
    alpha_threshold=0.05,
    use_mean=True,
    use_binary_weights=True,
    normalize_by_total=True,
    use_absolute=False
)

Predicting Pupil Rest and EEG

In [ ]:
arousal1_array = eeg_array_cut
fc1_array = eeg_fc_array_cut
arousal2_array = pupil_array_cut
fc2_array = pupil_fc_array_cut

predicted_output_path = "../4_results/2_classification/4_EEG_Rest_from_Pupil_Rest/EEG_Rest_from_Pupil_Rest_predicted.npy"
observed_output_path = "../4_results/2_classification/4_EEG_Rest_from_Pupil_Rest/EEG_Rest_from_Pupil_Rest_observed.npy"
features_output_path = "../4_results/2_classification/4_EEG_Rest_from_Pupil_Rest/EEG_Rest_from_Pupil_Rest_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



arousal2_array = eeg_array_cut
fc2_array = eeg_fc_array_cut
arousal1_array = pupil_array_cut
fc1_array = pupil_fc_array_cut

predicted_output_path = "../4_results/2_classification/5_Pupil_Rest_from_EEG_Rest/Pupil_Rest_from_EEG_Rest_predicted.npy"
observed_output_path = "../4_results/2_classification/5_Pupil_Rest_from_EEG_Rest/Pupil_Rest_from_EEG_Rest_observed.npy"
features_output_path = "../4_results/2_classification/5_Pupil_Rest_from_EEG_Rest/Pupil_Rest_from_EEG_Rest_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



Predicting Pupil Rest and Pupil Story

In [ ]:
arousal1_array = paranoia_array_cut
fc1_array = paranoia_fc_array_cut
arousal2_array = pupil_array_cut
fc2_array = pupil_fc_array_cut

predicted_output_path = "../4_results/2_classification/6_Pupil_Story_from_Pupil_Rest/Pupil_Story_from_Pupil_Rest_predicted.npy"
observed_output_path = "../4_results/2_classification/6_Pupil_Story_from_Pupil_Rest/Pupil_Story_from_Pupil_Rest_observed.npy"
features_output_path = "../4_results/2_classification/6_Pupil_Story_from_Pupil_Rest/Pupil_Story_from_Pupil_Rest_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



arousal2_array = paranoia_array_cut
fc2_array = paranoia_fc_array_cut
arousal1_array = pupil_array_cut
fc1_array = pupil_fc_array_cut

predicted_output_path = "../4_results/2_classification/7_Pupil_Rest_from_Pupil_Story/Pupil_Rest_from_Pupil_Story_predicted.npy"
observed_output_path = "../4_results/2_classification/7_Pupil_Rest_from_Pupil_Story/Pupil_Rest_from_Pupil_Story_observed.npy"
features_output_path = "../4_results/2_classification/7_Pupil_Rest_from_Pupil_Story/Pupil_Rest_from_Pupil_Story_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



Predicting EEG Rest and Pupil Story

In [ ]:
arousal1_array = paranoia_array_cut
fc1_array = paranoia_fc_array_cut
arousal2_array = eeg_array_cut
fc2_array = eeg_fc_array_cut

predicted_output_path = "../4_results/2_classification/8_Pupil_Story_from_EEG_Rest/Pupil_Story_from_EEG_Rest_predicted.npy"
observed_output_path = "../4_results/2_classification/8_Pupil_Story_from_EEG_Rest/Pupil_Story_from_EEG_Rest_observed.npy"
features_output_path = "../4_results/2_classification/8_Pupil_Story_from_EEG_Rest/Pupil_Story_from_EEG_Rest_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



arousal2_array = paranoia_array_cut
fc2_array = paranoia_fc_array_cut
arousal1_array = eeg_array_cut
fc1_array = eeg_fc_array_cut

predicted_output_path = "../4_results/2_classification/9_EEG_Rest_from_Pupil_Story/EEG_Rest_from_Pupil_Story_predicted.npy"
observed_output_path = "../4_results/2_classification/9_EEG_Rest_from_Pupil_Story/EEG_Rest_from_Pupil_Story_observed.npy"
features_output_path = "../4_results/2_classification/9_EEG_Rest_from_Pupil_Story/EEG_Rest_from_Pupil_Story_features.csv"

cross_modal_prediction(arousal1_array, fc1_array, arousal2_array, fc2_array,
                           predicted_output_path, observed_output_path, features_output_path, alpha_threshold = 0.05, 
                           use_mean=True, use_binary_weights=True,
                           normalize_by_total=True, use_absolute=False)



Movies and EEG Rest

In [ ]:
predicted_output_path = "../4_results/2_classification/10_EEG_Rest_from_Movies/EEG_Rest_from_Movies_predicted.npy"
observed_output_path = "../4_results/2_classification/10_EEG_Rest_from_Movies/EEG_Rest_from_Movies_observed.npy"
SR_to_other(eeg_array_cut, eeg_fc_array_cut, pop = predicted_output_path, oop = observed_output_path, use_absolute=False, normalize_by_total=True)

predicted_ff_output_path = "../4_results/2_classification/11_Movies_from_EEG_Rest/FilmFest_from_EEG_Rest_predicted.npy"
predicted_sher_output_path = "../4_results/2_classification/11_Movies_from_EEG_Rest/Sherlock_from_EEG_Rest_predicted.npy"
observed_ff_output_path = "../4_results/2_classification/11_Movies_from_EEG_Rest/FilmFest_from_EEG_Rest_observed.npy"
observed_sher_output_path = "../4_results/2_classification/11_Movies_from_EEG_Rest/Sherlock_from_EEG_Rest_observed.npy"
features_output_path = "../4_results/2_classification/11_Movies_from_EEG_Rest/Movies_from_EEG_Rest_features.csv"

other_to_SR(sherlock_fc = Sherlock_event_fc_array_cut, sherlock_beh = Sherlock_event_array_cut, filmfest_fc = FF_event_fc_array_cut, 
            filmfest_beh = FF_event_array_cut, other_fc = eeg_fc_array_cut, other_beh = eeg_array_cut, sherpop = predicted_sher_output_path, 
            sheroop = observed_sher_output_path, ffpop = predicted_ff_output_path, ffoop = observed_ff_output_path, features_output_path=features_output_path, 
            alpha_threshold=0.05, use_mean=True, use_binary_weights=True, use_absolute=False, normalize_by_total=True)


Movies and Pupil Rest

In [ ]:
predicted_output_path = "../4_results/2_classification/12_Pupil_Rest_from_Movies/Pupil_Rest_from_Movies_predicted.npy"
observed_output_path = "../4_results/2_classification/12_Pupil_Rest_from_Movies/Pupil_Rest_from_Movies_observed.npy"
SR_to_other(pupil_array_cut, pupil_fc_array_cut, pop = predicted_output_path, oop = observed_output_path, use_absolute=False, normalize_by_total=True)

predicted_ff_output_path = "../4_results/2_classification/13_Movies_from_Pupil_Rest/FilmFest_from_Pupil_Rest_predicted.npy"
predicted_sher_output_path = "../4_results/2_classification/13_Movies_from_Pupil_Rest/Sherlock_from_Pupil_Rest_predicted.npy"
observed_ff_output_path = "../4_results/2_classification/13_Movies_from_Pupil_Rest/FilmFest_from_Pupil_Rest_observed.npy"
observed_sher_output_path = "../4_results/2_classification/13_Movies_from_Pupil_Rest/Sherlock_from_Pupil_Rest_observed.npy"
features_output_path = "../4_results/2_classification/13_Movies_from_Pupil_Rest/Movies_from_Pupil_Rest_features.csv"

other_to_SR(sherlock_fc = Sherlock_event_fc_array_cut, sherlock_beh = Sherlock_event_array_cut, filmfest_fc = FF_event_fc_array_cut, 
            filmfest_beh = FF_event_array_cut, other_fc = pupil_fc_array_cut, other_beh = pupil_array_cut, sherpop = predicted_sher_output_path, 
            sheroop = observed_sher_output_path, ffpop = predicted_ff_output_path, ffoop = observed_ff_output_path, features_output_path=features_output_path, 
            alpha_threshold=0.05, use_mean=True, use_binary_weights=True, use_absolute=False, normalize_by_total=True)

Movies and Pupil Story

In [ ]:
predicted_output_path = "../4_results/2_classification/14_Pupil_Story_from_Movies/Pupil_Story_from_Movies_predicted.npy"
observed_output_path = "../4_results/2_classification/14_Pupil_Story_from_Movies/Pupil_Story_from_Movies_observed.npy"
SR_to_other(paranoia_array_cut, paranoia_fc_array_cut, pop = predicted_output_path, oop = observed_output_path, use_absolute=False, normalize_by_total=True)

predicted_ff_output_path = "../4_results/2_classification/15_Movies_from_Pupil_Story/FilmFest_from_Pupil_Story_predicted.npy"
predicted_sher_output_path = "../4_results/2_classification/15_Movies_from_Pupil_Story/Sherlock_from_Pupil_Story_predicted.npy"
observed_ff_output_path = "../4_results/2_classification/15_Movies_from_Pupil_Story/FilmFest_from_Pupil_Story_observed.npy"
observed_sher_output_path = "../4_results/2_classification/15_Movies_from_Pupil_Story/Sherlock_from_Pupil_Story_observed.npy"
features_output_path = "../4_results/2_classification/15_Movies_from_Pupil_Story/Movies_from_Pupil_Story_features.csv"

other_to_SR(sherlock_fc = Sherlock_event_fc_array_cut, sherlock_beh = Sherlock_event_array_cut, filmfest_fc = FF_event_fc_array_cut, 
            filmfest_beh = FF_event_array_cut, other_fc = paranoia_fc_array_cut, other_beh = paranoia_array_cut, sherpop = predicted_sher_output_path, 
            sheroop = observed_sher_output_path, ffpop = predicted_ff_output_path, ffoop = observed_ff_output_path, features_output_path=features_output_path, 
            alpha_threshold=0.05, use_mean=True, use_binary_weights=True, use_absolute=False, normalize_by_total=True)